# qwen_ft Vertex Training Submit and Eval

This notebook is a **thin wrapper** for steps 4-11 of
`docs/runbook-qwen-finetune-vertex.md`: it dry-runs and submits the Vertex AI
CustomJob that LoRA fine-tunes Qwen2.5-VL-7B, then runs the `qwen_ft`
evaluation through the public `aiforensics` CLI (eval-small first, eval-full
only after the parse-failure gate passes, then the report).

It does **not** train anything locally; training runs as a Vertex CustomJob on
an A100. Evaluation loads the trained adapter from
`gs://aiforensics-qwen-ft-579187260419/checkpoints/<protocol>/final_adapter`.

- Project: `579187260419`, location: `asia-southeast1`
- Enable **Internet**; provide GCP credentials via Application Default
  Credentials or the Kaggle Secret `GOOGLE_APPLICATION_CREDENTIALS`.

## 1. Install the repository and dependencies

The notebook clones this repository into writable storage and installs the
package. Nothing is pinned here; dependency names come from `pyproject.toml`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ai-image-forensics")
REPO_GIT_URL = "https://github.com/Nnguyen-dev2805/ai-image-forensics.git"

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_ROOT)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", "."],
    cwd=str(REPO_ROOT),
    check=True,
)
print("repository:", REPO_ROOT)

## 2. Verify GCP authentication

The submit script uses Application Default Credentials. If the Kaggle Secret
`GOOGLE_APPLICATION_CREDENTIALS` exists, it is activated first (credential
material is never printed); otherwise the cell verifies ADC is already usable.

In [ ]:
import json
import os
import shlex
import subprocess
import tempfile
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    service_account_json = UserSecretsClient().get_secret("GOOGLE_APPLICATION_CREDENTIALS")
    service_account_info = json.loads(service_account_json)
    key_file = Path(tempfile.mkstemp(prefix="sa-", suffix=".json")[1])
    key_file.write_text(service_account_json, encoding="utf-8")
    os.chmod(key_file, 0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(key_file)
    activate_cmd = f"gcloud auth activate-service-account --key-file={key_file}"
    subprocess.run(shlex.split(activate_cmd), check=True)
    print("gcloud authenticated as:", service_account_info.get("client_email", "<unknown>"))
except Exception:
    print("no Kaggle secret found; relying on existing Application Default Credentials")

probe = subprocess.run(
    ["gcloud", "auth", "list", "--format=value(account)"],
    capture_output=True,
    text=True,
    check=True,
)
print("active accounts:", probe.stdout.strip() or "<none>")

## 3. Dry-run the Vertex CustomJob payload

Prints the exact REST payload that would be submitted: A100 40GB worker,
GCS checkpoint prefix, and the LoRA training flags. Inspect it before
submitting; training costs real money.

In [ ]:
import shlex
import subprocess

SUBMIT_CMD = (
    f"{sys.executable} scripts/submit_qwen_ft_vertex_job.py "
    f"--project-id 579187260419 "
    f"--location asia-southeast1 "
    f"--bucket-uri gs://aiforensics-qwen-ft-579187260419 "
    "--protocol protocol-a-small "
    "--machine-type a2-highgpu-1g "
    "--accelerator-type NVIDIA_TESLA_A100 "
    "--accelerator-count 1"
)

subprocess.run(shlex.split(SUBMIT_CMD + " --dry-run"), cwd=str(REPO_ROOT), check=True)

## 4. Submit the Vertex CustomJob

Submits the job and prints the resource name. Monitor progress with:

```bash
gcloud ai custom-jobs describe <JOB_NAME> --region=asia-southeast1
gcloud ai custom-jobs tail-logs <JOB_NAME> --region=asia-southeast1
```

Checkpoints land under
`{BUCKET_URI}/checkpoints/protocol-a-small/`; the eval cells below require
`final_adapter/adapter_config.json` to exist there.

In [ ]:
import shlex
import subprocess

result = subprocess.run(
    shlex.split(SUBMIT_CMD),
    cwd=str(REPO_ROOT),
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(
    "Training runs remotely. Re-run this notebook from section 5 once "
    "final_adapter/adapter_config.json exists in GCS."
)

## 5. Generate the runtime eval configs

The committed `configs/qwen_ft_protocol_a_small.yaml` is a read-only template.
This cell writes eval-small and eval-full copies under
`.cache/aiforensics-notebook/` that only relocate the evaluation manifest and
the writable roots. The eval-full manifest is the operator-provided
7000-image all-val CSV produced by the earlier Vertex all-val flow.

In [ ]:
import yaml
from pathlib import Path

TEMPLATE = REPO_ROOT / "configs/qwen_ft_protocol_a_small.yaml"
CACHE_NOTEBOOK = REPO_ROOT / ".cache/aiforensics-notebook"

# Operator inputs: writable roots and the full-validation manifest.
OUTPUT_ROOT = Path("/kaggle/working/outputs-qwen-ft")
CACHE_ROOT = Path("/kaggle/working/cache-qwen-ft")
EVAL_FULL_MANIFEST = Path("/kaggle/working/manifests/genimage_all_val.csv")

with open(TEMPLATE, encoding="utf-8") as handle:
    base_cfg = yaml.safe_load(handle)


def write_eval_config(suffix: str, manifest: Path, report_name: str) -> Path:
    cfg = yaml.safe_load(yaml.safe_dump(base_cfg))
    cfg["paths"]["manifest_root"] = str(manifest.parent)
    cfg["paths"]["cache_root"] = str(CACHE_ROOT / suffix)
    cfg["paths"]["output_root"] = str(OUTPUT_ROOT / suffix)
    cfg["datasets"]["tiny_genimage"]["train_manifest"] = str(manifest.parent / "train.csv")
    cfg["datasets"]["tiny_genimage"]["dev_manifest"] = str(manifest)
    cfg["datasets"]["genimage_unseen"]["manifest"] = str(manifest)
    cfg["report"]["filename"] = report_name
    out = CACHE_NOTEBOOK / f"qwen_ft_protocol_a_{suffix}.yaml"
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as handle:
        yaml.safe_dump(cfg, handle, sort_keys=False)
    return out


EVAL_SMALL_CONFIG = write_eval_config(
    "eval-small", Path("/kaggle/working/manifests/eval_small.csv"), "qwen_ft_eval_small_report.md"
)
EVAL_FULL_CONFIG = write_eval_config(
    "eval-full", EVAL_FULL_MANIFEST, "qwen_ft_eval_full_report.md"
)

for path in (EVAL_SMALL_CONFIG, EVAL_FULL_CONFIG):
    print("runtime config:", path)

## 6. Run qwen_ft eval-small

Loads the base model plus the trained LoRA adapter and emits label-only
predictions for the 700-image eval-small manifest through the public CLI.

In [ ]:
import shlex
import subprocess

subprocess.run(
    shlex.split(f"aiforensics run --baseline qwen_ft --config {EVAL_SMALL_CONFIG}"),
    cwd=str(REPO_ROOT),
    check=True,
)
subprocess.run(
    shlex.split(f"aiforensics evaluate --config {EVAL_SMALL_CONFIG}"),
    cwd=str(REPO_ROOT),
    check=True,
)

## 7. Parse-failure gate

Eval-full costs 7000 inferences; it only runs when the eval-small parse
failure rate is below the 2% success criterion from the design spec.

In [ ]:
import json
from pathlib import Path


def latest_qwen_ft_predictions(output_root: Path) -> Path:
    runs = sorted(
        (path for path in output_root.iterdir() if path.is_dir() and "_qwen_ft" in path.name),
        key=lambda path: path.name,
    )
    if not runs:
        raise FileNotFoundError(f"No qwen_ft run directory under {output_root}")
    return runs[-1] / "predictions.jsonl"


pred_path = latest_qwen_ft_predictions(OUTPUT_ROOT / "eval-small")
records = [json.loads(line) for line in pred_path.read_text(encoding="utf-8").splitlines()]
total = len(records)
failed = sum(1 for record in records if record.get("parse_status") == "failed")
parse_failure_rate = failed / total if total else 1.0

print(f"parse failure rate: {failed}/{total} = {parse_failure_rate:.4f}")
if parse_failure_rate >= 0.02:
    raise RuntimeError(
        "Parse failure rate is at or above the 2% success criterion; fix the "
        "adapter/parser before spending eval-full inference budget."
    )
print("GATE OK: eval-full may run.")

## 8. Run qwen_ft eval-full and generate reports

Runs the full 7000-image validation manifest, evaluates both manifests, and
writes the comparison reports next to the run artifacts.

In [ ]:
import shlex
import subprocess

subprocess.run(
    shlex.split(f"aiforensics run --baseline qwen_ft --config {EVAL_FULL_CONFIG}"),
    cwd=str(REPO_ROOT),
    check=True,
)
for config in (EVAL_SMALL_CONFIG, EVAL_FULL_CONFIG):
    subprocess.run(
        shlex.split(f"aiforensics report --config {config}"),
        cwd=str(REPO_ROOT),
        check=True,
    )

## 9. Artifacts

```text
gs://aiforensics-qwen-ft-579187260419/data/protocol-a-small/...   exported data
gs://aiforensics-qwen-ft-579187260419/checkpoints/protocol-a-small/...  checkpoints
<OUTPUT_ROOT>/<suffix>/<run_id>/predictions.jsonl
<OUTPUT_ROOT>/<suffix>/<run_id>/metrics.json
<OUTPUT_ROOT>/<suffix>/<run_id>/confusion_matrix.json
<OUTPUT_ROOT>/<suffix>/<configured report filename>
```

Keep every artifact: a run without saved artifacts is not a valid experiment
result per the design spec.